# noise-ceiling-mol на Colab
Конвейер: данные ChEMBL → стандартизация → очистка от бенчмарков → потолок шума → модели → доля потолка.
**Не проверено на Colab** (писалось и отлаживалось в другой среде). Все шаги идут по реальным данным ChEMBL через REST API;
если API не отвечает, шаг падает с ошибкой, данные не имитируются. Время работы — часы, включите Runtime → Change runtime type.
Репозиторий публичный? Если нет, в ячейке клонирования подставьте токен GitHub.

In [ ]:
# 1. Установка. torch на Colab уже есть
!pip install -q rdkit scikit-learn pandas scipy pyyaml transformers pytest

In [ ]:
# 2. Клонирование и установка пакета
import os
if not os.path.exists('noise-ceiling-mol'):
    !git clone -b cloud-data https://github.com/Ruskinkot1/noise-ceiling-mol.git
%cd noise-ceiling-mol
!pip install -q -e .
!OMP_NUM_THREADS=1 python -m pytest -q

In [ ]:
# 3. Параметры
import os
os.environ['OMP_NUM_THREADS'] = '1'   # иначе sklearn GBM очень медленный при нескольких процессах
PILOT_TASKS = 'CHEMBL237_IC50 CHEMBL206_IC50 CHEMBL204_Ki'

## Данные

In [ ]:
# 4. Выгрузка активностей из ChEMBL (29 мишеней из configs/endpoints.yaml; займёт десятки минут)
!python scripts/01_extract_chembl.py

In [ ]:
# 5. Стандартизация RDKit, одно значение на (молекула, документ); печатает число задач выше порога
!python scripts/02_standardize.py | tail -5

In [ ]:
# 6. Реестр уже виденных молекул (MoleculeNet, TDC) и очистка. Polaris требует входа и не скачивается.
!python scripts/build_registry.py
# --no-year: без фильтра по году документа (с ним остаётся 8249 молекул и 0 задач; решение по фильтру — за исследователем)
!python scripts/02b_decontaminate.py --no-year --n-jobs 2 | tail -15

In [ ]:
# 7. Аудит валидности данных (дубли, большие расхождения между документами, сдвиги по документам)
!python scripts/02c_audit_data.py

## Пилот: 3 задачи, очищенные и неочищенные данные

In [ ]:
# 8. Подготовка пилота и эмбеддинги ChemBERTa-77M и MoLFormer (скрипты считают на CPU, GPU не используется)
!python scripts/make_pilot.py --tasks $PILOT_TASKS
!python scripts/06_embed_foundation.py --processed data/pilot_raw --out-dir data/pilot_emb --models chemberta-77m-mlm molformer-xl-10pct

In [ ]:
# 9. Потолок шума с бутстрэпом
!python scripts/03_estimate_ceiling.py --processed data/pilot_clean --out results/pilot/ceil_clean --n-boot 1000
!python scripts/03_estimate_ceiling.py --processed data/pilot_raw   --out results/pilot/ceil_raw   --n-boot 1000

In [ ]:
# 10. Модели x 3 сплита x сиды (простые модели 5 сидов, GNN 3 сида; подбор 3 конфигурации на модель).
# Займёт часы; прогресс в results/pilot/*.log
!bash scripts/run_pilot.sh

In [ ]:
# 11. Объединение результатов, доля потолка, парные сравнения, разрезы
import glob, pandas as pd
for v in ['clean', 'raw']:
    files = [f for f in glob.glob(f'results/pilot/{v}_*_cheap.csv') + glob.glob(f'results/pilot/{v}_*_gnn.csv')]
    pred = [f for f in glob.glob(f'results/pilot/{v}_*.pred.csv')]
    os.makedirs(f'results/pilot_{v}', exist_ok=True)
    pd.concat([pd.read_csv(f) for f in files]).to_csv(f'results/pilot_{v}/benchmark.csv', index=False)
    pd.concat([pd.read_csv(f) for f in pred]).to_csv(f'results/pilot_{v}/predictions.csv', index=False)
    pd.read_csv(f'results/pilot/ceil_{v}/ceilings.csv').to_csv(f'results/pilot_{v}/ceilings.csv', index=False)
!python scripts/05_fraction_of_ceiling.py --results results/pilot_clean
!python scripts/07_slices.py --results results/pilot_clean --processed data/pilot_clean

In [ ]:
# 12. Сохранить результаты на Google Drive (или скачать архивом)
!zip -qr results.zip results configs docs/*.md
from google.colab import files
files.download('results.zip')